# NLLB-600M: бенчмарк на T4 GPU

**Цель:** измерить скорость перевода 1000 запросов EN→RU с fp16 на T4, чтобы понять, хватит ли времени на 100k.

**Пороги:**
- ≥9.3 req/s → 100k влезут в 3 часа ✅
- 7–9.2 req/s → 100k влезут в 4 часа (рискованно для Colab free) ⚠️
- <7 req/s → нужно снижать beams с 4 до 2 ❌

## 1. Проверка GPU

In [ ]:
!nvidia-smi

**Убедись, что Runtime → Change runtime type → T4 GPU**

## 2. Монтируем Google Drive (кеш переживёт разрыв сессии)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Клонируем репозиторий

In [ ]:
!git clone https://github.com/Krynovv/kry_prompt-lib.git
%cd kry_prompt-lib/kursovaya-tex

## 4. Устанавливаем зависимости

In [ ]:
!pip install -q torch sentencepiece sacremoses transformers

## 5. Настраиваем окружение

In [ ]:
import os
import sys

# Путь кеша в Google Drive (переживёт разрыв сессии)
os.environ['SEMSEARCH_CACHE_DIR'] = '/content/drive/MyDrive/semsearch_cache'

# Добавляем текущую директорию в PYTHONPATH для импорта модулей
sys.path.insert(0, '.')

print(f"Кеш будет сохраняться в: {os.environ["SEMSEARCH_CACHE_DIR"]}")
print(f"Рабочая директория: {os.getcwd()}")

## 6. Запускаем бенчмарк (1000 запросов EN→RU)

In [ ]:
!python -m code.semsearch.data.translate \
  --model nllb-en2ru \
  --input code/semsearch/data/train_queries.txt \
  --output /tmp/benchmark_en2ru.txt \
  --device cuda \
  --dtype float16 \
  --benchmark 1000

## 7. Анализ результата

In [ ]:
import json

with open('/tmp/benchmark_en2ru.txt') as f:
    lines = f.readlines()
    summary = json.loads(lines[-1])

req_per_sec = summary['sentences_per_sec']
time_for_100k_hours = 100000 / req_per_sec / 3600

print(f"\n{'='*60}")
print(f"РЕЗУЛЬТАТ БЕНЧМАРКА")
print(f"{'='*60}")
print(f"Скорость: {req_per_sec:.2f} req/s")
print(f"Время на 100k: {time_for_100k_hours:.2f} часов")
print(f"{'='*60}\n")

if req_per_sec >= 9.3:
    print("✅ ОТЛИЧНО: 100k влезут в 3 часа. Запускай полный прогон!")
elif req_per_sec >= 7.0:
    print("⚠️ РИСКОВАННО: 100k влезут в 3–4 часа. Можно попробовать, но есть риск обрыва.")
elif req_per_sec >= 4.0:
    print("❌ МЕДЛЕННО: нужно 4–7 часов. Запусти бенчмарк с beams=2 (добавь --beams 2).")
else:
    print("❌ ОЧЕНЬ МЕДЛЕННО: >7 часов. Либо beams=1, либо нужен другой подход.")

print(f"\nПодробности: {summary}")

## Что дальше?

**Если скорость ≥7 req/s:**
1. Открой ноутбук `full_run_100k.ipynb`
2. Запусти полный прогон 100k (EN→RU + RU→EN)

**Если <7 req/s:**
1. Повтори бенчмарк с `--beams 2` (вместо 4)
2. Если и тогда медленно — пиши руководителю